# 02 · Churn & Tenure Definitions

**Purpose:** Define churn and tenure precisely, show why a plain "% churned" is misleading, and build the monthly churn rate (life table) that the rest of the project uses.

**Input:** `data/processed/subscriptions_clean.csv`
**Output:** `data/processed/subscriptions_model.csv`, `reports/life_table_overall.csv`

**Key definitions**
- **Event:** customer cancelled (`event = 1`); still active at 2025-12-31 is *censored* (`event = 0`).
- **Tenure:** days from signup to cancel date, or to 2025-12-31 if active.
- **Tenure month:** 30-day blocks since signup (0 = first 30 days).
- **Monthly churn rate:** cancellations in a tenure month ÷ customers at risk in that month.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

PROC = Path("../data/processed")
REPORTS = Path("../reports")
DATA_END = pd.Timestamp("2025-12-31")   # last day covered by the data

subs = pd.read_csv(PROC / "subscriptions_clean.csv", parse_dates=["signup_date", "churn_date"])
print(subs.shape)
subs.head(3)

(4200, 14)


,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason,usage_months_expected,usage_months_observed,usage_stopped_early
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,True,2025-06-29,technical_issues,10,10,False
1,2,2025-08-14,annual,basic,9.74,promo_offer,web,45-54,False,NaT,NaN,5,5,False
2,3,2025-01-24,monthly,basic,12.99,referral,ios,55+,False,NaT,NaN,12,1,True


In [2]:
subs["event"] = subs.churned.astype(int)                      # 1 = cancelled, 0 = still active (censored)
subs["end_date"] = subs.churn_date.fillna(DATA_END)           # cancel date, or end of data if active
subs["tenure_days"] = (subs.end_date - subs.signup_date).dt.days
subs["duration_days"] = subs.tenure_days.clip(lower=1)        # same-day cancels count as 1 day
subs["tenure_month"] = subs.duration_days // 30               # 0 = first 30 days, 1 = days 30-59, ...

subs[["customer_id", "signup_date", "end_date", "event",
      "tenure_days", "duration_days", "tenure_month"]].head()

,customer_id,signup_date,end_date,event,tenure_days,duration_days,tenure_month
0,1,2024-09-11,2025-06-29,1,291,291,9
1,2,2025-08-14,2025-12-31,0,139,139,4
2,3,2025-01-24,2025-12-31,0,341,341,11
3,4,2024-08-15,2024-08-20,1,5,5,0
4,5,2025-06-24,2025-12-31,0,190,190,6


Sanity-checking tenure: no negatives, and comparing churners with active customers.

In [3]:
print("tenure_days < 0:", (subs.tenure_days < 0).sum())
print("longest possible tenure:", (DATA_END - subs.signup_date.min()).days, "days")
print()
print(subs.groupby("churned").tenure_days.describe().round(1))

tenure_days < 0: 0
longest possible tenure: 548 days

          count   mean    std   min    25%    50%    75%    max
churned                                                        
False    2432.0  247.1  128.6  62.0  137.0  229.0  333.0  548.0
True     1768.0  156.2  124.1   0.0   52.0  114.0  278.2  531.0


Looking at the same-day cancellations and confirming each gets a duration of 1 day.

In [4]:
same_day = subs[subs.tenure_days == 0]
print(len(same_day), "same-day cancellations")
same_day[["customer_id", "signup_date", "churn_date", "tenure_days", "duration_days", "plan_type"]]

7 same-day cancellations


,customer_id,signup_date,churn_date,tenure_days,duration_days,plan_type
535,536,2025-03-03,2025-03-03,0,1,monthly
751,752,2024-09-27,2024-09-27,0,1,monthly
1243,1244,2025-04-20,2025-04-20,0,1,annual
1685,1686,2024-07-02,2024-07-02,0,1,monthly
3023,3024,2025-06-01,2025-06-01,0,1,monthly
3859,3860,2025-02-19,2025-02-19,0,1,monthly
4187,4188,2024-08-28,2024-08-28,0,1,monthly


Showing why a plain "% churned" is misleading. Newer signups have had less time to cancel (censoring).

In [5]:
subs["signup_quarter"] = subs.signup_date.dt.to_period("Q").astype(str)

naive = subs.groupby("signup_quarter").agg(
    customers=("customer_id", "size"),
    pct_churned=("churned", "mean"),
    max_days_observed=("tenure_days", "max"),
)
naive["pct_churned"] = (naive.pct_churned * 100).round(1)
naive

,customers,pct_churned,max_days_observed
signup_quarter,,,
2024Q3,594,64.3,548
2024Q4,786,63.9,456
2025Q1,784,43.2,364
2025Q2,857,34.0,274
2025Q3,876,23.6,183
2025Q4,303,15.5,91


Checking when annual and monthly customers cancel, to see whether annual plans need special handling.

In [6]:
churners = subs[subs.churned].copy()
churners["tenure_band"] = pd.cut(
    churners.tenure_days,
    bins=[-1, 30, 90, 180, 270, 365, 600],
    labels=["0-30", "31-90", "91-180", "181-270", "271-365", "366+"],
)

print("Number of churners")
print(pd.crosstab(churners.tenure_band,churners.plan_type))
print("\n% of each plan's churners")
print(pd.crosstab(churners.tenure_band, churners.plan_type, normalize="columns").mul(100).round(1))

Number of churners
plan_type    annual  monthly
tenure_band                 
0-30             15      258
31-90            22      463
91-180           31      331
181-270          23      164
271-365         313       99
366+              2       47

% of each plan's churners
plan_type    annual  monthly
tenure_band                 
0-30            3.7     18.9
31-90           5.4     34.0
91-180          7.6     24.3
181-270         5.7     12.0
271-365        77.1      7.3
366+            0.5      3.5


Defining a reusable function that calculates the churn rate per 30-day tenure month, adjusted for censoring.

In [10]:
def life_table(df, duration="duration_days", event="event", block=30):
    d = df[[duration, event]].copy()
    d["k"] = d[duration] // block                      # tenure month in which each customer exits or is last seen
    rows = []
    for k in range(int(d.k.max()) + 1):
        entered = (d.k >= k).sum()                     # customers who reached the start of month k
        churned = ((d.k == k) & (d[event] == 1)).sum() # cancelled during month k
        censored = ((d.k == k) & (d[event] == 0)).sum()# still active but last seen in month k
        at_risk = entered - censored / 2               # censored customers count as half a month of exposure
        rate = churned / at_risk if at_risk > 0 else np.nan
        rows.append((k, entered, churned, censored, at_risk, rate))
    lt = pd.DataFrame(rows, columns=["tenure_month", "entered", "churned", "censored", "at_risk", "churn_rate"])
    lt["survival"] = (1 - lt.churn_rate.fillna(0)).cumprod()   # share still subscribed after this month
    return lt


Computing the overall monthly churn rate by tenure month, our main definition.

In [11]:
lt = life_table(subs)
lt.round({"at_risk": 1, "churn_rate": 3, "survival": 3})

,tenure_month,entered,churned,censored,at_risk,churn_rate,survival
0,0,4200,267,0,4200.0,0.064,0.936
1,1,3933,246,0,3933.0,0.063,0.878
2,2,3687,239,239,3567.5,0.067,0.819
3,3,3209,164,242,3088.0,0.053,0.776
4,4,2803,114,227,2689.5,0.042,0.743
5,5,2462,85,195,2364.5,0.036,0.716
6,6,2182,78,198,2083.0,0.037,0.689
7,7,1906,69,184,1814.0,0.038,0.663
8,8,1653,44,184,1561.0,0.028,0.644
9,9,1425,52,169,1340.5,0.039,0.619


Comparing monthly and annual plans: the share still subscribed after 90, 180 and 360 days.

In [13]:
rows = []
for plan, g in subs.groupby("plan_type"):
    t = life_table(g).set_index("tenure_month")
    rows.append({
        "plan_type": plan,
        "cutomers": len(g),
        "retianed_after_90d": t.survival.loc[2],
        "retianed_after_180d": t.survival.loc[5],
        "retianed_after_360d": t.survival.loc[11],
    })
pd.DataFrame(rows).round(3)

,plan_type,cutomers,retianed_after_90d,retianed_after_180d,retianed_after_360d
0,annual,1539,0.976,0.951,0.411
1,monthly,2661,0.728,0.582,0.402


Building the alternative definition for the sensitivity check, treating the 139 flagged customers as churned at the end of their signup month.

In [15]:
flag = subs.usage_stopped_early
alt_end = subs.signup_date + pd.offsets.MonthEnd(0)

subs["event_alt"] = np.where(flag, 1, subs.event)
subs["end_date_alt"] = subs.end_date.where(~flag, alt_end)
subs["duration_days_alt"] = (subs.end_date_alt - subs.signup_date).dt.days.clip(lower= 1)

print("churners, main definition:", subs.event.sum())
print("churners, alternative definition:", subs.event_alt.sum())

churners, main definition: 1768
churners, alternative definition: 1907


Comparing the two definitions side by side to see whether the picture changes.

In [16]:
main = life_table(subs)
alt = life_table(subs, duration="duration_days_alt", event="event_alt")

compare = pd.DataFrame({
    "rate_main": main.churn_rate,
    "rate_alt": alt.churn_rate,
    "survival_main": main.survival,
    "survival_alt": alt.survival,
})

print(compare.head(8))
print("\n retained after 90 / 180 / 360 days")
for name, t in [("main", main), ("alt ", alt)]:
    print(name, t.survival.loc[[2, 5, 11]].round(3).tolist())

   rate_main  rate_alt  survival_main  survival_alt
0   0.063571  0.096667       0.936429      0.903333
1   0.062548  0.064839       0.877857      0.844762
2   0.066994  0.069659       0.819046      0.785917
3   0.053109  0.055405       0.775548      0.742373
4   0.042387  0.044323       0.742674      0.709468
5   0.035948  0.037694       0.715977      0.682725
6   0.037446  0.039324       0.689166      0.655878
7   0.038037  0.040000       0.662952      0.629642

 retained after 90 / 180 / 360 days
main [0.819, 0.716, 0.403]
alt  [0.786, 0.683, 0.37]


Saving the model-ready table and the overall life table for the next notebooks.

In [17]:
subs.to_csv(PROC / "subscriptions_model.csv", index=False)
life_table(subs).to_csv(REPORTS / "life_table_overall.csv", index=False)

print("subscriptions_model:", subs.shape)
print("new columns:", [c for c in subs.columns if c not in pd.read_csv(PROC / "subscriptions_clean.csv", nrows=1).columns])

subscriptions_model: (4200, 23)
new columns: ['event', 'end_date', 'tenure_days', 'duration_days', 'tenure_month', 'signup_quarter', 'event_alt', 'end_date_alt', 'duration_days_alt']


**Results**
- **Overall retention:**  81.9% after 90 days, 71.6% after 180, 40.3% after 360.
- **Censoring**  makes raw cohort churn (64% down to 15.5%) meaningless, so from now on we use monthly rates by tenure.
- **Annual plans**  churn at renewal (77% of their churners leave at 271–365 days), so every comparison must be stratified by plan type.
- **The 139 flagged customers**  affect only month-0 churn, and the alternative columns (*_alt) are ready for Phase 4's sensitivity check.